# 02 — The Born Rule on Real Hardware

In `01/04` you sampled the Born rule on a simulator; now you read it straight off a real QPU and look for the device's read-out bias.

**Prerequisites:** `01/12_shot_noise_and_first_circuit`, `05/01_using_a_real_quantum_computer`.

**What you'll be able to do**
- Run a single-qubit state through `SamplerV2` and read its measurement counts back as probabilities.
- Tell a read-out lean from shot noise by setting it against $\sigma = \sqrt{p(1-p)/\mathrm{shots}}$.
- Watch statistical scatter shrink as you add shots, and test whether a systematic bias is left once it has.
- Calibrate the read-out with `|0>` and `|1>`, and correct a result by inverting the $2 \times 2$ confusion matrix.

You have already derived the Born rule and watched shot noise on the simulator, where the answer was exact. Here the same `|+>` state runs on a superconducting chip, and the counts come back with whatever fingerprint the device leaves: shot noise always, and possibly a small *systematic* lean toward one outcome. Telling those two apart is the skill this notebook builds. A lean of that kind is one of the imperfect outcomes that the density matrices of `01/13` are made to describe, and section 3 runs the standard correction for the part of it that the read-out causes. The `USE_HARDWARE` switch keeps everything runnable on your laptop today and reaches a real device the moment you flip it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qot_course.hardware.records import save_run_record
from qot_course.hardware.runtime import select_backend
from qot_course.quantum.states import born_probabilities, KET_PLUS

np.random.seed(42)
viz.use_course_style()

In [ ]:
USE_HARDWARE = False  # flip to True after you've set up credentials (see 05/01)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")
if not is_real:
    backend.set_options(seed_simulator=42)  # fixed seed: every offline number below is reproducible

## 1 · |+> on the device

We build `|+>` with a single Hadamard, run it through the `SamplerV2` primitive, and compare the device's counts to the ideal Born probabilities. The state `|+> = (|0> + |1>)/sqrt(2)` is the cleanest possible test: textbook quantum mechanics says a measurement in the computational basis lands on `0` and `1` with probability exactly one-half each. Any departure we see is the device speaking, not the mathematics.

In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)                      # |0> -> |+>
qc.measure(0, 0)
isa = generate_preset_pass_manager(backend=backend, optimization_level=1).run(qc)
SHOTS = 4096
job = SamplerV2(mode=backend).run([isa], shots=SHOTS)
counts = job.result()[0].data.c.get_counts()
save_run_record(counts, notebook="05_02", tag="plus", device=backend.name, shots=SHOTS,
                job_id=job.job_id() if is_real else None, is_real=is_real)

ideal = born_probabilities(KET_PLUS)           # {"0": 0.5, "1": 0.5}
total = sum(counts.values())
measured = {k: counts.get(k, 0) / total for k in ("0", "1")}
sigma_shot = np.sqrt(0.5 * 0.5 / SHOTS)        # binomial standard deviation of P(0) for a fair coin
print("ideal   :", ideal)
print("measured:", {k: round(v, 4) for k, v in measured.items()})
print(f"shot noise of P(0) at {SHOTS} shots: sigma = sqrt(0.25 / {SHOTS}) = {sigma_shot:.4f}")
print(f"P(0) - 0.5 = {measured['0'] - 0.5:+.4f}, that is {(measured['0'] - 0.5) / sigma_shot:+.2f} sigma")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
x = np.arange(2); w = 0.38
ax.bar(x - w / 2, [ideal["0"], ideal["1"]], width=w, label="ideal", color=COLORS["muted"])
ax.bar(x + w / 2, [measured["0"], measured["1"]], width=w, label="measured", color=COLORS["quantum"])
ax.axhline(0.5, color=COLORS["highlight"], linestyle="--", label="fair 50/50")
ax.set_xticks(x); ax.set_xticklabels(["0", "1"])
ax.set_ylabel("probability")
ax.set_title("Born rule for |+>: ideal vs a real device", pad=12)
ax.legend()
plt.show()

**Read the figure.** On the offline model the measured split is $0.4917$ / $0.5083$: P(0) sits $0.0083$ below one half, which is about one standard deviation of the shot noise of 4,096 shots ($\sigma = 0.0078$, printed above). Sampling alone produces gaps of that size, so this one run does not show a read-out lean; the shot sweep and the calibration below test for one properly. Offline, a lean could only come from the *modeled* FakeMarrakesh noise; flip `USE_HARDWARE=True` and the same cell reports the literal device.

### A real run on IBM hardware

On **`ibm_marrakesh`** (156-qubit Heron processor, 2026-05-30), measuring `|+>` returned **P(0) = 0.491, P(1) = 0.509**. With the 4,096 shots of the cell above, that is $1.15$ standard deviations below one half (next cell), and sampling alone departs that far about one time in four. This single run therefore does not establish a read-out lean. The job identifier and the raw counts of that run were not kept, so the value cannot be recomputed from this repository. Set `USE_HARDWARE = True` to reproduce on the least-busy QPU your account can reach.

In [ ]:
# The real run quoted above, set against the shot noise of 4096 shots (the shot count of the cell above).
from math import erfc, sqrt
REAL_P0 = 0.491   # ibm_marrakesh, 2026-05-30; raw counts not kept
z_real = (REAL_P0 - 0.5) / sigma_shot
print(f"real run: P(0) - 0.5 = {REAL_P0 - 0.5:+.3f} = {z_real:+.2f} sigma")
print(f"probability that shot noise alone departs at least that far (two-sided): {erfc(abs(z_real) / sqrt(2)):.2f}")

## 2 · Shot noise on hardware

Re-run the same `|+>` circuit at increasing shot counts. Two effects can pull on the result. The *statistical* deviation from 0.5, the scatter you met as shot noise in `01/12`, shrinks like $1/\sqrt{\mathrm{shots}}$ as you average over more measurements. A *systematic* lean, if the device has one, does not shrink: once the scatter falls below it, the mean P(0) stops approaching one half. One run per shot count cannot tell these apart, so we repeat each shot count 20 times with fixed seeds. The spread of the 20 values measures the scatter, and their mean, with its standard error, tests for a lean.

In [ ]:
shot_grid = [256, 1024, 4096, 16384]
N_REP = 20   # repeats of each shot count; their spread IS the shot noise
pubs = [(isa, None, shots) for shots in shot_grid for _ in range(N_REP)]
sweep_job = SamplerV2(mode=backend).run(pubs)   # one job holds every repeat
sweep = sweep_job.result()
sweep_counts = {f"shots{pubs[i][2]}_rep{i % N_REP:02d}": sweep[i].data.c.get_counts() for i in range(len(pubs))}
save_run_record(sweep_counts, notebook="05_02", tag="shot_sweep", device=backend.name,
                shots=sum(p[2] for p in pubs), job_id=sweep_job.job_id() if is_real else None,
                is_real=is_real, extra={"shots_is": "total over all pubs", "shot_grid": shot_grid, "n_rep": N_REP})

p0 = np.array([sweep[i].data.c.get_counts().get("0", 0) / pubs[i][2] for i in range(len(pubs))])
p0 = p0.reshape(len(shot_grid), N_REP)
mean, spread = p0.mean(axis=1), p0.std(axis=1, ddof=1)
sem = spread / np.sqrt(N_REP)                       # standard error of the mean P(0)
theory = np.sqrt(0.25 / np.array(shot_grid))        # binomial sigma for P(0) = 0.5, not fitted
print(" shots   mean P(0)   spread   sqrt(0.25/shots)   mean - 0.5   (+/- s.e.)")
for s, m, sd, th, se in zip(shot_grid, mean, spread, theory, sem):
    print(f"{s:6d}    {m:.4f}    {sd:.4f}        {th:.4f}         {m - 0.5:+.4f}    ({se:.4f})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.loglog(shot_grid, spread, "o", color=COLORS["quantum"], label=f"spread of P(0) over {N_REP} repeats")
ax1.loglog(shot_grid, theory, "--", color=COLORS["flow"], label=r"$\sqrt{0.25/\mathrm{shots}}$ (no fit)")
ax1.set_xlabel("shots"); ax1.set_ylabel("standard deviation of P(0)")
ax1.set_title("Statistical scatter falls with shots", pad=10); ax1.legend()
ax2.errorbar(shot_grid, mean - 0.5, yerr=sem, fmt="o", color=COLORS["quantum"], capsize=4,
             label="mean P(0) - 0.5, +/- standard error")
ax2.axhline(0.0, color=COLORS["highlight"], linestyle="--", label="no bias")
ax2.set_xscale("log"); ax2.set_xlabel("shots"); ax2.set_ylabel("mean P(0) - 0.5")
ax2.set_title("Is there a bias left once the scatter is averaged?", pad=10); ax2.legend()
fig.tight_layout()
plt.show()

**Read the figure.** On the left, the spread of P(0) over the 20 repeats falls from $0.0319$ at 256 shots to $0.0039$ at 16,384 shots. It follows the guide $\sqrt{0.25/\mathrm{shots}}$, a prediction and not a fit, within the scatter that 20 repeats allow; the 4,096-shot point sits highest, $0.0102$ against $0.0078$. On the right, the mean of P(0) − 0.5 stays within one standard error of zero at every shot count, and at 16,384 shots it is $-0.0002 \pm 0.0009$. So the offline model shows no bias floor at this resolution: its whole departure from one half is shot noise. Section 3 shows why, and what a real device can add.

## 3 · Calibrating the read-out, and correcting for it

The read-out of a qubit can be measured on its own. Prepare `|0>` and read it many times: the fraction of `1`s is the error rate $e_0 = P(\text{read } 1 \mid \text{prepared } 0)$. Prepare `|1>` with an `x` gate: the fraction of `0`s is $e_1 = P(\text{read } 0 \mid \text{prepared } 1)$. The two rates form the confusion matrix

$$M = \begin{pmatrix} 1 - e_0 & e_1 \\ e_0 & 1 - e_1 \end{pmatrix},$$

whose column $j$ says how a prepared `|j>` is read. The meter turns the true probabilities $p$ into $Mp$, so solving $Mp = $ (measured) recovers $p$. This is read-out error correction, the one error mitigation this module runs. It corrects the meter only: gate errors and decoherence before the measurement stay in the result. For `|+>` the rates predict $P(0) = 0.5 + (e_1 - e_0)/2$, so a lean appears only when the two rates differ. We calibrate the same physical qubit that measured `|+>` above.

In [ ]:
q = isa.layout.final_index_layout()[0]          # the physical qubit that measured |+> above
pm_q = generate_preset_pass_manager(backend=backend, optimization_level=1, initial_layout=[q])
cal_circuits = []
for prepared in (0, 1):
    c = QuantumCircuit(1, 1)
    if prepared:
        c.x(0)                               # |1>
    c.measure(0, 0)
    cal_circuits.append(pm_q.run(c))
CAL_SHOTS = 16384
cal_job = SamplerV2(mode=backend).run(cal_circuits, shots=CAL_SHOTS)
cal_res = cal_job.result()
cal_counts = {f"prep{p}": cal_res[p].data.c.get_counts() for p in (0, 1)}
save_run_record(cal_counts, notebook="05_02", tag="readout_calibration", device=backend.name, shots=CAL_SHOTS,
                job_id=cal_job.job_id() if is_real else None, is_real=is_real, extra={"physical_qubit": int(q)})

e0 = cal_counts["prep0"].get("1", 0) / CAL_SHOTS   # P(read 1 | prepared 0)
e1 = cal_counts["prep1"].get("0", 0) / CAL_SHOTS   # P(read 0 | prepared 1)
se = lambda p: np.sqrt(p * (1 - p) / CAL_SHOTS)
print(f"physical qubit {q}, {CAL_SHOTS} shots per calibration circuit")
print(f"  P(read 1 | prepared 0) = {e0:.4f} +/- {se(e0):.4f}")
print(f"  P(read 0 | prepared 1) = {e1:.4f} +/- {se(e1):.4f}")
print(f"lean these rates predict for |+>: P(0) = 0.5 + (e1 - e0)/2 = {0.5 + (e1 - e0) / 2:.4f}")

M = np.array([[1 - e0, e1],
              [e0, 1 - e1]])                 # column j: how a prepared |j> is read
raw = np.array([measured["0"], measured["1"]])
corrected = np.linalg.solve(M, raw)          # invert the read-out: the probabilities before the meter
print(f"|+> before correction: P(0) = {raw[0]:.4f}   ({(raw[0] - 0.5) / sigma_shot:+.2f} sigma from 0.5)")
print(f"|+> after  correction: P(0) = {corrected[0]:.4f}   ({(corrected[0] - 0.5) / sigma_shot:+.2f} sigma from 0.5)")
if not is_real:
    from qiskit_ibm_runtime import fake_provider
    snapshot = fake_provider.FakeMarrakesh().properties().qubit_property(q)
    print(f"calibration snapshot of qubit {q}: P(read 1 | 0) = {snapshot['prob_meas1_prep0'][0]:.4f}, "
          f"P(read 0 | 1) = {snapshot['prob_meas0_prep1'][0]:.4f}, "
          f"read-out error used by the noise model = {backend.target['measure'][(q,)].error:.4f}")
    lean_snapshot = (snapshot['prob_meas0_prep1'][0] - snapshot['prob_meas1_prep0'][0]) / 2
    print(f"lean the snapshot rates would predict for |+>: P(0) = {0.5 + lean_snapshot:.4f}, "
          f"{lean_snapshot / sigma_shot:.2f} sigma of {SHOTS} shots")

**Read the output.** On the offline model, qubit 0 misreads `|0>` with probability $0.0119$ and `|1>` with probability $0.0124$, equal within their error bars. Equal rates predict no lean ($P(0) = 0.5002$), and the correction agrees: P(0) for `|+>` moves from $0.4917$ to $0.4912$, far less than the shot noise of $0.0078$, and the result is consistent with one half before and after. The last lines say why. The calibration snapshot of FakeMarrakesh records two different rates for this qubit, $0.0093$ and $0.0132$, but the noise model built from it applies one read-out error, $0.0112$, to both outcomes, so the offline twin cannot show an asymmetric lean. A real device has two rates that can differ. Had they been the snapshot's, the predicted lean would be $P(0) = 0.5020$, a quarter of the shot noise of 4,096 shots: visible only with many more shots, and then removed by this correction. Flip `USE_HARDWARE=True` and the same cell calibrates and corrects the real qubit.

## Your turn

- Replace `qc.h(0)` with `qc.ry(theta, 0)` for a `theta` of your choice and compare the measured `P(0)` to the prediction `cos**2(theta/2)` you derived in `01/04`. How close does the device land, and does the gap grow as you tilt the state?
- Flip `USE_HARDWARE=True` (with credentials set per `05/01`) and re-run. The bias you see is now the real chip's, not a model's.
- Print `isa.layout` to see which physical qubit actually ran your circuit. Choose a different qubit on the device and check whether the read-out bias changes — read-out error is per-qubit, so it often does.

## What you built

- A real-hardware Born-rule run end to end: build `|+>`, transpile it to an ISA circuit, sample it with `SamplerV2`, and read the counts back as probabilities.
- A side-by-side picture of ideal vs measured that makes read-out bias visible as a departure from the fair 50/50 line.
- A repeated shot sweep that measures the statistical scatter, which averages away, and tests for a systematic bias floor, which would not (none at the resolution of the offline model).
- A read-out calibration of one qubit, its two error rates, and a correction of the `|+>` result by inverting the confusion matrix: the module's one error mitigation.

## What's next

Next you go beyond single probabilities and reconstruct a full quantum state on hardware — **tomography** — then score how close the reconstructed state is to the ideal with **fidelity**, turning the bias you met here into a quantitative measure.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 2 — the Born rule and measurement (Cambridge University Press, 2010).
- Qiskit IBM Runtime documentation — `SamplerV2`, transpilation, and reading counts: <https://docs.quantum.ibm.com>.

**Previous:** `notebooks/05_RealQuantumHardware/01_using_a_real_quantum_computer.ipynb` · **Next:** `notebooks/05_RealQuantumHardware/03_tomography_and_fidelity_on_real_hardware.ipynb`